# River Ice Classification with Deep Learning
## Hands-on Workshop

---

### Background

Monitoring river ice is critical for flood prediction, water resource management,
and ecological research.  Fixed cameras on bridges provide a cheap, continuous
data stream — but produce far more images than humans can label by hand.

In this workshop we train a **convolutional neural network (CNN)** to automatically
classify each camera frame as **Open Water** or **Ice Presence**.

---

### Learning Objectives

By the end of this notebook you will be able to:

1. Load and explore an image classification dataset
2. Visualise the effect of data augmentation
3. Explain **transfer learning** and why it matters
4. Build a classifier by adapting a pre-trained ImageNet backbone
5. Train the model and interpret loss / accuracy curves
6. Evaluate performance with a confusion matrix

### File structure

```
train_FLOW_WorkShop/
    classification_train_V2.ipynb <- this notebook
    utils.py                      <- all helper code
    requirements.txt
```

---
## Step 0 - Configuration

**Edit the cell below** to set your data path and experiment settings.
Every other cell reads from these variables.

In [ ]:
# ==========================================
# WORKSHOP SETUP: RUN THIS FIRST
# ==========================================
import os
# Download the dataset directly using gdown (No login required)
DATASET_FILE_ID = '1tr20pAzMODuH3pOV1DmSollL3YuYMV55'
WEIGHTS_FILE_ID = '1lcb0F_SDsNJLb-IM4VQbYWA6D2xrM-jB'
if not os.path.exists("workshop_data.zip"):
  print("📥 Downloading dataset...")
  !gdown {DATASET_FILE_ID} -O workshop_data.zip
  !gdown {WEIGHTS_FILE_ID} -O pretrained_weights.pth

  print("📦 Extracting files to local SSD...")
  # !unzip -q workshop_data.zip -d /content/

  print("✅ Setup complete! Data is ready at '/content/'")


In [ ]:
pip install torchmetrics


In [ ]:
# ------------------------------------------------------------------
# >>> CHANGE THIS to the folder that contains  train/  and  test/
# ------------------------------------------------------------------
import os
DATASET_PATH = "\content\dataset"

# --- Model ---
# Options: "resnet18" | "resnet34" | "resnet50" | "resnet101" | "efficientnet_b0"
MODEL_NAME = "resnet18"
FREEZE     = True     # True  = only final layer trained (fast, less data needed)
                      # False = all layers fine-tuned   (slower, more data needed)
use_pretrained = True  # due to the workshop time limit, we will use a pretrained weights,
                       # but you can set this to False to train from scratch (not recommended for small datasets)

# --- Training ---
INIT_LR      = 1e-4
UPDATE_LR    = 0.1    # update learning rate after epochs > PATIENCE
PATIENCE     = 5      # patience to update LR or for early stopping
NUM_EPOCHS   = 50
BATCH_SIZE   = 16
VALID_SPLIT  = 0.2    # fraction of training data used for validation
RANDOM_STATE = 42

# --- Dataset ---
CLASSES      = ["Open Water", "Ice Presence"]
INPUT_WIDTH  = 1152
INPUT_HEIGHT = 640

# --- Output paths (created automatically) ---
OUTPUT_DIR = os.path.join("outputs", MODEL_NAME + "_workshop")
MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# --- Derived paths ---
TRAIN_IMAGES = os.path.join(DATASET_PATH, "train", "images")
TRAIN_CSV    = os.path.join(DATASET_PATH, "train", "classes_all.csv")
TEST_IMAGES  = os.path.join(DATASET_PATH, "test",  "images")
TEST_CSV     = os.path.join(DATASET_PATH, "test",  "classes_all.csv")

print("Configuration loaded.")
print(f"  Dataset : {DATASET_PATH}")
print(f"  Model   : {MODEL_NAME}  (freeze={FREEZE})")
print(f"  Epochs  : {NUM_EPOCHS},  batch: {BATCH_SIZE},  lr: {INIT_LR}")


---
## Step 1 - Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import torchvision.models as tv_models
from torchmetrics.classification import MulticlassAccuracy
from sklearn.model_selection import train_test_split
from imutils import paths

# All workshop helpers live in a single utils module
from utils import (
    Dataset,
    get_training_augmentation,
    visualize,
    TrainEpoch,
    ValidEpoch,
    test_model,
    show_confusion_matrix,
    EarlyStopping,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")
if DEVICE == "cpu":
    print("  No GPU detected - training will work but will be slower.")


---
## Step 2 - Load and Explore the Dataset

| Split | Purpose |
|---|---|
| **Training** | Model learns from these images |
| **Validation** | Monitors overfitting during training (no weight updates) |
| **Test** | Held out completely; used only for final evaluation |

We start with a pre-defined train/test split and carve a validation set out
of the training portion (20% by default).

In [ ]:
# Load image paths
all_train_paths = sorted(list(paths.list_images(TRAIN_IMAGES)))
all_test_paths  = sorted(list(paths.list_images(TEST_IMAGES)))

# Load CSV label files
train_df = pd.read_csv(TRAIN_CSV, header=None, names=["name", "label"])
test_df  = pd.read_csv(TEST_CSV,  header=None, names=["name", "label"])

def get_labels(image_paths, label_df):
    lookup = label_df.set_index("name")["label"].to_dict()
    return [lookup[os.path.basename(p)] for p in image_paths]

# Train / validation split
train_paths, valid_paths = train_test_split(
    all_train_paths, test_size=VALID_SPLIT, random_state=RANDOM_STATE
)
train_labels = get_labels(train_paths,    train_df)
valid_labels = get_labels(valid_paths,    train_df)
test_labels  = get_labels(all_test_paths, test_df)

print(f"Classes          : {CLASSES}")
print(f"Training images  : {len(train_paths)}")
print(f"Validation images: {len(valid_paths)}")
print(f"Test images      : {len(all_test_paths)}")

unique, counts = np.unique(train_labels, return_counts=True)
print("\nTraining class distribution:")
for idx, cnt in zip(unique, counts):
    print(f"  {CLASSES[idx]}: {cnt} images ({100*cnt/len(train_labels):.1f}%)")


In [ ]:
preview = Dataset(train_paths, train_labels)
print("Sample training images (no augmentation):")
for idx in [0, 1, 2]:
    image, label = preview[idx]
    visualize(**{f"Class: {CLASSES[label]}": image})

---
## Step 3 - Data Augmentation

Augmentation artificially expands the training set by applying random but
realistic transforms: flips, brightness shifts, blur, noise, perspective.
This forces the model to become **invariant** to irrelevant variations and
reduces **overfitting**.

> Augmentation is applied **only during training** so evaluation stays fair.

In [ ]:
aug_preview = Dataset(train_paths, train_labels,
                      augmentation=get_training_augmentation())

print("The same source image after three different random augmentations:")
for _ in range(3):
    image, label = aug_preview[3]
    visualize(**{f"Augmented - {CLASSES[label]}": image})


---
## Step 4 - Build the Model (Transfer Learning from ImageNet)

Training a CNN from scratch on a small dataset leads to overfitting.
**Transfer learning** avoids this by starting from a network already trained
on **ImageNet** (1.2 M images, 1000 classes).  The backbone has already
learned edges, textures, and shapes that transfer well to almost any task.

```
ImageNet pre-training (done)          Our fine-tuning
──────────────────────────────        ──────────────────────────────
ResNet-50 backbone                    Same backbone  +  new head
  -> edges, textures, shapes            -> freeze backbone (or unfreeze)
                                        -> train fc layer only
                                        -> output: Open Water / Ice Presence
```

| MODEL_NAME | Params | Notes |
|---|---|---|
| `resnet18` | 11 M | Fast; good for small datasets |
| `resnet50` | 25 M | Strong all-round baseline |
| `resnet101` | 44 M | Higher capacity; risks overfitting |
| `efficientnet_b0` | 5 M | Modern, efficient architecture |

In [ ]:
num_classes = len(CLASSES)

MODEL_REGISTRY = {
    "resnet18":        (tv_models.resnet18,        tv_models.ResNet18_Weights.DEFAULT),
    "resnet34":        (tv_models.resnet34,        tv_models.ResNet34_Weights.DEFAULT),
    "resnet50":        (tv_models.resnet50,         tv_models.ResNet50_Weights.DEFAULT),
    "resnet101":       (tv_models.resnet101,        tv_models.ResNet101_Weights.DEFAULT),
    "efficientnet_b0": (tv_models.efficientnet_b0,  tv_models.EfficientNet_B0_Weights.DEFAULT),
}

if use_pretrained:
    print(f"Loading pre-trained weights from 'pretrained_weights.pth'...")
    model_fn, _ = MODEL_REGISTRY[MODEL_NAME]
    model = model_fn()
    if hasattr(model, "fc"):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        model.classifier[-1] = nn.Linear(
            model.classifier[-1].in_features, num_classes)
    model.load_state_dict(torch.load("pretrained_weights.pth", map_location="cpu"))
else:
    model_fn, weights = MODEL_REGISTRY[MODEL_NAME]
    model = model_fn(weights=weights)

    # Replace the final layer to match our number of classes
    if hasattr(model, "fc"):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    print(f"Loaded '{MODEL_NAME}' with ImageNet pre-trained weights.")

model = model.to(DEVICE)
total = sum(p.numel() for p in model.parameters())


In [ ]:
total = sum(p.numel() for p in model.parameters())

if FREEZE:
    for name, param in model.named_parameters():
        if "fc" not in name and "classifier" not in name:
            param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Frozen parameters    : {total-trainable:,}  ({100*(total-trainable)/total:.2f}%)")
if FREEZE:
    print("\nOnly the final classification layer will be updated.")
    print("Set FREEZE = False in Step 0 to fine-tune the entire network.")


In [ ]:
# Optional model summary (install with: pip install torchsummary)
try:
    from torchsummary import summary
    summary(model, (3, INPUT_HEIGHT, INPUT_WIDTH))
except ImportError:
    print("torchsummary not installed. Install with: pip install torchsummary")


---
## Step 5 - Training Setup

| Component | Choice | Why |
|---|---|---|
| **Loss** | Cross-Entropy | Standard for multi-class classification |
| **Metric** | Accuracy | Fraction of correctly classified images |
| **Optimizer** | Adam | Adaptive learning rate; fast to converge |

In [ ]:
loss_fn  = nn.CrossEntropyLoss()

accuracy = MulticlassAccuracy(num_classes=num_classes).to(DEVICE)
accuracy.name = "Accuracy"   # key used in the training log dict

optimizer = optim.Adam(model.parameters(), lr=INIT_LR)

train_dataset = Dataset(train_paths, train_labels,
                        augmentation=get_training_augmentation(), normalize=True)
valid_dataset = Dataset(valid_paths, valid_labels, normalize=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

train_epoch = TrainEpoch(model, loss_fn, [accuracy], optimizer, device=DEVICE)
valid_epoch = ValidEpoch(model, loss_fn, [accuracy],             device=DEVICE)

print(f"Training batches   : {len(train_loader)}")
print(f"Validation batches : {len(valid_loader)}")


In [ ]:
stopper = EarlyStopping(patience=2*PATIENCE, min_delta=0.005)

---
## Step 6 - Train the Model

Each epoch:
1. **Training pass** - processes every batch, computes loss, updates weights
2. **Validation pass** - no weight updates; measures generalisation
3. Best model checkpoint is saved whenever validation accuracy improves

> If training loss falls but validation loss rises, the model is **overfitting**.

In [ ]:
if use_pretrained:
    print(f"Evaluating pre-trained model on validation set...")
    valid_logs = valid_epoch.run(valid_loader)
    print(f"  Validation accuracy: {valid_logs['Accuracy']:.4f}")
    
else:
    print("No pre-trained weights loaded, starting training from scratch.")
    best_val_acc   = 0.0
    train_loss_log = []
    train_acc_log  = []
    valid_loss_log = []
    valid_acc_log  = []

    count = 0

    for epoch in range(NUM_EPOCHS):
        count += 1
        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
        train_logs = train_epoch.run(train_loader)
        valid_logs = valid_epoch.run(valid_loader)

        train_loss_log.append(train_logs["loss"])
        train_acc_log.append(train_logs["Accuracy"])
        valid_loss_log.append(valid_logs["loss"])
        valid_acc_log.append(valid_logs["Accuracy"])

        if valid_logs["Accuracy"] > best_val_acc:
            count = 0
            best_val_acc = valid_logs["Accuracy"]
            torch.save(model.state_dict(), MODEL_PATH)
            print(f"  --> Best model saved  (val acc = {best_val_acc:.4f})")

        if count >= PATIENCE:
            count = 0
            for g in optimizer.param_groups:
                g["lr"] *= UPDATE_LR
            print(f"  --> Learning rate reduced to {g['lr']}")

        stopper.plateau(valid_logs["loss"])
        if stopper.early_stop:
            print("  --> Early stopping")
            break

    print(f"\nTraining complete.  Best validation accuracy: {best_val_acc:.4f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(train_loss_log) + 1)

ax1.semilogy(epochs, train_loss_log, label="Training")
ax1.semilogy(epochs, valid_loss_log, "--", label="Validation")
ax1.set(xlabel="Epoch", ylabel="Loss (log scale)", title="Loss")
ax1.legend(); ax1.grid(True)

ax2.plot(epochs, [a * 100 for a in train_acc_log], label="Training")
ax2.plot(epochs, [a * 100 for a in valid_acc_log], "--", label="Validation")
ax2.set(xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy")
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "training_curves.png"), dpi=150)
plt.show()


### Interpreting Training Curves

| Pattern | What it means |
|---|---|
| Both curves decrease together | Healthy training |
| Val loss rises while train loss falls | **Overfitting** |
| Both plateau early | Learning rate may be too low |
| Training loss oscillates | Learning rate may be too high |

---
## Step 7 - Evaluate on the Test Set

The test set was **never seen** during training or validation, giving an
honest estimate of real-world performance.

The **confusion matrix** shows — for each true class — what fraction of
samples were correctly classified vs. misclassified.

In [ ]:
# Reconstruct the same architecture and load the best saved weights
if not use_pretrained:
    model_fn, _ = MODEL_REGISTRY[MODEL_NAME]
    best_model  = model_fn()
    if hasattr(best_model, "fc"):
        best_model.fc = nn.Linear(best_model.fc.in_features, num_classes)
    else:
        best_model.classifier[-1] = nn.Linear(
            best_model.classifier[-1].in_features, num_classes)
    best_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
    print(f"Best {MODEL_NAME} loaded from {MODEL_PATH}")
else:
    best_model = model


In [ ]:
test_dataset     = Dataset(all_test_paths, test_labels, normalize=True)
test_dataset_vis = Dataset(all_test_paths, test_labels)
test_loader      = DataLoader(test_dataset, batch_size=1)
class_to_idx     = {cls: idx for idx, cls in enumerate(CLASSES)}

preds, labels, logits, probs, incorrect = test_model(
    best_model, test_loader, test_dataset_vis, class_to_idx, plot_incorrect=False
)

print(f"\nTest accuracy : {(preds == labels).mean() * 100:.2f}%")
print(f"Misclassified : {len(incorrect)} / {len(labels)} images")


In [ ]:
show_confusion_matrix(
    preds, labels, class_to_idx,
    save_path=os.path.join(PLOT_DIR, "confusion_matrix.png"),
)


---
## Discussion Questions

1. **Frozen vs fine-tuned**: Set `FREEZE = False` in Step 0, retrain, and compare.
   What changed? Why?

2. **Backbone choice**: Change `MODEL_NAME` to `"resnet18"` or `"efficientnet_b0"`.
   Does accuracy change? How does training speed compare?

3. **Learning rate**: Try `INIT_LR = 0.001` and `INIT_LR = 0.00001`.
   How do the training curves differ?

4. **Augmentation ablation**: In Step 5, set `augmentation=None`.
   Does the gap between training and validation accuracy widen? Why?

5. **Class imbalance**: If one class had 90% of images and the other 10%,
   what problem could arise? How might you fix it?
   *(Hint: look up `torch.nn.CrossEntropyLoss(weight=...)`)*